# Sierra Leone Solar Farm Data - Exploratory Data Analysis

This notebook performs comprehensive EDA on solar farm data from Sierra Leone (Bumbuna).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/sierraleone-bumbuna.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Summary Statistics & Missing Values

In [ ]:
df.describe()

In [ ]:
missing_values = df.isna().sum()
missing_percent = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing_values, 'Percentage': missing_percent})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Percentage', ascending=False)
print("\nColumns with missing values:")
print(missing_df)

print("\nColumns with >5% missing values:")
print(missing_df[missing_df['Percentage'] > 5])

## 3. Outlier Detection (Z-score Method)

In [ ]:
numeric_cols = ['GHI', 'DNI', 'DHI', 'ModA', 'ModB', 'WS', 'WSgust']
outlier_counts = {}

for col in numeric_cols:
    if col in df.columns:
        z_scores = np.abs(stats.zscore(df[col].dropna()))
        outliers = (z_scores > 3).sum()
        outlier_counts[col] = outliers

outlier_df = pd.DataFrame.from_dict(outlier_counts, orient='index', columns=['Outlier_Count'])
print("Outliers detected (|Z| > 3):")
print(outlier_df)

## 4. Data Cleaning

In [ ]:
df_clean = df.copy()

df_clean['Timestamp'] = pd.to_datetime(df_clean['Timestamp'])

key_columns = ['GHI', 'DNI', 'DHI']
df_clean = df_clean.dropna(subset=key_columns)

numeric_cols_all = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols_all:
    if df_clean[col].isna().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {df_clean.shape}")
print(f"Rows removed: {df.shape[0] - df_clean.shape[0]}")

## 5. Time Series Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(df_clean['Timestamp'], df_clean['GHI'], alpha=0.7, linewidth=0.5)
axes[0, 0].set_title('GHI Over Time')
axes[0, 0].set_ylabel('GHI (W/m²)')

axes[0, 1].plot(df_clean['Timestamp'], df_clean['DNI'], alpha=0.7, linewidth=0.5, color='orange')
axes[0, 1].set_title('DNI Over Time')
axes[0, 1].set_ylabel('DNI (W/m²)')

axes[1, 0].plot(df_clean['Timestamp'], df_clean['DHI'], alpha=0.7, linewidth=0.5, color='green')
axes[1, 0].set_title('DHI Over Time')
axes[1, 0].set_ylabel('DHI (W/m²)')

axes[1, 1].plot(df_clean['Timestamp'], df_clean['Tamb'], alpha=0.7, linewidth=0.5, color='red')
axes[1, 1].set_title('Temperature Over Time')
axes[1, 1].set_ylabel('Temperature (°C)')

plt.tight_layout()
plt.show()

In [ ]:
df_clean['Month'] = df_clean['Timestamp'].dt.month
df_clean['Hour'] = df_clean['Timestamp'].dt.hour

monthly_avg = df_clean.groupby('Month')[['GHI', 'DNI', 'DHI']].mean()

fig, ax = plt.subplots(figsize=(12, 6))
monthly_avg.plot(kind='bar', ax=ax)
ax.set_title('Monthly Average Solar Irradiance')
ax.set_xlabel('Month')
ax.set_ylabel('Irradiance (W/m²)')
plt.xticks(rotation=0)
plt.legend()
plt.show()

## 6. Cleaning Impact Analysis

In [ ]:
if 'Cleaning' in df_clean.columns:
    cleaning_impact = df_clean.groupby('Cleaning')[['ModA', 'ModB']].mean()
    print("Module Performance by Cleaning Status:")
    print(cleaning_impact)
    
    cleaning_impact.plot(kind='bar', figsize=(10, 6))
    plt.title('Module Performance: Before (0) vs After (1) Cleaning')
    plt.xlabel('Cleaning Status')
    plt.ylabel('Average Module Reading (W/m²)')
    plt.xticks(rotation=0)
    plt.legend(['Module A', 'Module B'])
    plt.show()

## 7. Correlation Analysis

In [ ]:
corr_cols = ['GHI', 'DNI', 'DHI', 'TModA', 'TModB', 'Tamb', 'RH', 'WS', 'BP']
available_corr_cols = [col for col in corr_cols if col in df_clean.columns]

plt.figure(figsize=(12, 10))
corr_matrix = df_clean[available_corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 8. Scatter Plots - Relationship Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(df_clean['WS'], df_clean['GHI'], alpha=0.3)
axes[0, 0].set_xlabel('Wind Speed (m/s)')
axes[0, 0].set_ylabel('GHI (W/m²)')
axes[0, 0].set_title('Wind Speed vs GHI')

axes[0, 1].scatter(df_clean['RH'], df_clean['Tamb'], alpha=0.3, color='orange')
axes[0, 1].set_xlabel('Relative Humidity (%)')
axes[0, 1].set_ylabel('Temperature (°C)')
axes[0, 1].set_title('Humidity vs Temperature')

axes[1, 0].scatter(df_clean['RH'], df_clean['GHI'], alpha=0.3, color='green')
axes[1, 0].set_xlabel('Relative Humidity (%)')
axes[1, 0].set_ylabel('GHI (W/m²)')
axes[1, 0].set_title('Humidity vs GHI')

axes[1, 1].scatter(df_clean['WD'], df_clean['GHI'], alpha=0.3, color='red')
axes[1, 1].set_xlabel('Wind Direction (°)')
axes[1, 1].set_ylabel('GHI (W/m²)')
axes[1, 1].set_title('Wind Direction vs GHI')

plt.tight_layout()
plt.show()

## 9. Wind Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(df_clean['WD'], df_clean['WS'], alpha=0.3)
axes[0].set_xlabel('Wind Direction (°)')
axes[0].set_ylabel('Wind Speed (m/s)')
axes[0].set_title('Wind Direction vs Wind Speed')

axes[1].hist(df_clean['WS'], bins=50, edgecolor='black')
axes[1].set_xlabel('Wind Speed (m/s)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Wind Speed Distribution')

plt.tight_layout()
plt.show()

## 10. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df_clean['GHI'], bins=50, edgecolor='black')
axes[0, 0].set_xlabel('GHI (W/m²)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('GHI Distribution')

axes[0, 1].hist(df_clean['DNI'], bins=50, edgecolor='black', color='orange')
axes[0, 1].set_xlabel('DNI (W/m²)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('DNI Distribution')

axes[1, 0].hist(df_clean['DHI'], bins=50, edgecolor='black', color='green')
axes[1, 0].set_xlabel('DHI (W/m²)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('DHI Distribution')

axes[1, 1].hist(df_clean['Tamb'], bins=50, edgecolor='black', color='red')
axes[1, 1].set_xlabel('Temperature (°C)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Temperature Distribution')

plt.tight_layout()
plt.show()

## 11. Temperature & Humidity Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(df_clean['RH'], df_clean['Tamb'], alpha=0.3)
axes[0].set_xlabel('Relative Humidity (%)')
axes[0].set_ylabel('Ambient Temperature (°C)')
axes[0].set_title('RH vs Temperature')

axes[1].scatter(df_clean['RH'], df_clean['GHI'], alpha=0.3, color='orange')
axes[1].set_xlabel('Relative Humidity (%)')
axes[1].set_ylabel('GHI (W/m²)')
axes[1].set_title('RH vs Solar Radiation')

plt.tight_layout()
plt.show()

## 12. Bubble Chart

In [ ]:
sample_data = df_clean.sample(min(1000, len(df_clean)))

plt.figure(figsize=(12, 8))
plt.scatter(sample_data['GHI'], sample_data['Tamb'], 
           s=sample_data['RH']*2, alpha=0.5, c=sample_data['RH'], cmap='viridis')
plt.xlabel('GHI (W/m²)')
plt.ylabel('Temperature (°C)')
plt.title('GHI vs Temperature (bubble size = Relative Humidity)')
plt.colorbar(label='Relative Humidity (%)')
plt.show()

## 13. Save Cleaned Data

In [ ]:
df_clean.to_csv('../data/sierraleone_clean.csv', index=False)
print(f"Cleaned data saved to ../data/sierraleone_clean.csv")
print(f"Shape: {df_clean.shape}")

## Key Insights

1. **Data Quality**: [Add observations about missing values and outliers]
2. **Solar Radiation Patterns**: [Add observations about GHI, DNI, DHI trends]
3. **Environmental Factors**: [Add observations about temperature, humidity, wind]
4. **Cleaning Impact**: [Add observations about module performance]
5. **Correlations**: [Add key correlation findings]